# 14 - Analog-propagation baseline (C1 / C2)

1. analog neighbors: fragment + neutral-loss binned cosine over the whole frozen reference library (sparse product; CUDA optional), modified-cosine rescoring (existing implementation). Excluded: the query itself, T1 duplicates, **every reference of the fold's hidden C2/C3 connectivities**.
2. candidates: universe mass index at `analog.candidate_ppm`, fold view (C3 truths removed).
3. features: Tanimoto propagation from analog molecules (`casmi.analog.propagation`).
4. evaluation: heuristic scorers + a small LightGBM lambdarank (existing `casmi.ranking.lambdamart` harness, connectivity-grouped folds) with an ablation.

GPU optional (prefilter). Everything expensive is cached / resumable on Drive. Set `QUICK_RUN_MAX_QUERIES_PER_FOLD` for a fast first pass.

In [ ]:
import os, sys, json, time
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', str(DRIVE_ROOT / 'repo')))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.environment import ensure_packages, set_seeds, system_report, gpu_report
ensure_packages({'rdkit': 'rdkit', 'lightgbm': 'lightgbm', 'yaml': 'pyyaml'})
from casmi.workspace.config import load_v2_config, input_path
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '14_colab_analog_baseline'
QUICK_RUN_MAX_QUERIES_PER_FOLD = 2000   # None = every C1/C2/C3 query (deterministic subsample otherwise)
RUN_TAG = 'analog_v1' + ('_quick' if QUICK_RUN_MAX_QUERIES_PER_FOLD else '')
print(system_report()); print(gpu_report())

In [ ]:
import numpy as np, pandas as pd
from casmi.workspace.artifacts import read_metadata, write_metadata
from casmi.validation.regimes import stable_unit_hash
REG = pd.read_parquet(P.processed_dir / 'validation_regimes.parquet')
REG_META = read_metadata(P.processed_dir, 'validation_regimes.metadata.json')
if REG_META['c2_mode'] != 'external_universe':
    print('WARNING: PRELIMINARY regimes -- re-run notebook 11 (REBUILD = True) after notebook 12')
dev = pd.read_parquet(input_path(CFG, P, 'dev_queries'), columns=['query_id', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type'])
dev['query_id'] = dev['query_id'].astype(str)
Q = REG.dropna(subset=['regime']).merge(dev, on='query_id', how='left', validate='one_to_one')
if QUICK_RUN_MAX_QUERIES_PER_FOLD:
    Q['_h'] = stable_unit_hash(Q['query_id'].tolist(), f"{CFG['validation']['split_seed']}|quick")
    Q = Q.sort_values('_h').groupby('fold', group_keys=False).head(QUICK_RUN_MAX_QUERIES_PER_FOLD).drop(columns='_h')
Q = Q.sort_values(['fold', 'query_id']).reset_index(drop=True)
display(Q.groupby(['fold', 'regime']).size().unstack(fill_value=0))

In [ ]:
from casmi.analog.retrieval import AnalogConfig, AnalogLibrary, AnalogSearcher
ACFG = AnalogConfig.from_dict(CFG['analog'])
t0 = time.time()
ALIB = AnalogLibrary(P.bundle_dir)
rows = ALIB.rows_of_spectrum_ids(Q['query_id'])
print(f'library rows: {ALIB.n_refs:,} | query ids found as library spectrum ids: {(rows >= 0).mean():.4f}')
assert (rows >= 0).all(), 'dev query ids must be library (training) spectrum ids -- check the id mapping before continuing'
LF, LN = ALIB.binned_matrices(ACFG, cache_dir=P.cache_dir)
SEARCHER = AnalogSearcher(ALIB, LF, LN, ACFG)
print(f'binned library: fragments nnz={LF.nnz:,}, neutral losses nnz={LN.nnz:,} | device={SEARCHER.device} | {time.time() - t0:.0f}s')

In [ ]:
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.candidates.universe import load_candidate_keys
from casmi.chemistry.fingerprint_store import FingerprintCache
UNIVERSE_ROOT = P.candidate_db_dir / 'universe'
IDX = CandidateMassIndex.load(UNIVERSE_ROOT / 'index')
UKEYS = load_candidate_keys(UNIVERSE_ROOT)
FP = FingerprintCache(P.cache_dir / 'fingerprints', CFG['analog']['fingerprint']['radius'], CFG['analog']['fingerprint']['n_bits'])
print(f'universe candidates: {IDX.n_candidates:,} | cached fingerprints: {len(FP):,}')

In [ ]:
# per-fold neighbors + features (resumable shards on Drive)
from casmi.analog.pipeline import run_analog_fold
NB_DIR = P.cache_dir / 'analog_neighbors' / RUN_TAG
FEAT_DIR = P.processed_dir / 'analog_features' / RUN_TAG
RUNS = []
for f in sorted(Q['fold'].unique()):
    t0 = time.time()
    rec = run_analog_fold(f, Q, REG, ALIB, SEARCHER, IDX, UNIVERSE_ROOT, UKEYS, FP, CFG['analog'], NB_DIR, FEAT_DIR)
    rec['seconds'] = time.time() - t0
    RUNS.append(rec); print(rec)
FP.write_meta()
RUNS = pd.DataFrame(RUNS)

In [ ]:
from casmi.analog.features import ANALOG_FEATURES, MASS_FEATURES, feature_schema
from casmi.analog.pipeline import load_features
F = load_features(FEAT_DIR)
F = F[F['query_id'].isin(set(Q['query_id']))].reset_index(drop=True)
pos = F.groupby('query_id')['is_true_candidate'].sum()
per_regime = Q.assign(n_pos=Q['query_id'].map(pos).fillna(0)).groupby('regime')['n_pos'].agg(['mean', 'max'])
display(per_regime)
assert (Q.loc[Q.regime == 'C3', 'query_id'].map(pos).fillna(0) == 0).all(), 'C3 truths must never be candidates'
assert pos.max() <= 1, 'at most one positive (connectivity-level) per query'
print(f'feature rows: {len(F):,} | schema: {feature_schema()["feature_hash"]}')

In [ ]:
# heuristic (untrained) scorers
from casmi.analog.pipeline import per_query_ranks
from casmi.validation.metrics import ranking_metrics, composite_mrr
F['score_mass_only'] = -F['abs_mass_error_ppm']
SCORERS = {'mass_only': 'score_mass_only', 'analog_propagation': 'analog_propagation_score', 'analog_max_weighted': 'analog_max_weighted'}
PER_QUERY, ROWS = {}, []
for name, col in SCORERS.items():
    pq = per_query_ranks(F, col, Q)
    PER_QUERY[name] = pq
    for reg, g in pq.groupby('regime'):
        ROWS.append({'variant': name, 'kind': 'heuristic', 'regime': reg, **ranking_metrics(g)})
display(pd.DataFrame(ROWS))

In [ ]:
# small LightGBM lambdarank baseline + ablation (existing harness; model f never sees fold-f connectivities)
from casmi.ranking.lambdamart import LGBM_PARAMS, connectivity_disjointness, fit_fold_models, predict_by_fold, save_fold_models
VARIANTS = {'lgbm_mass_only': MASS_FEATURES, 'lgbm_analog_only': ANALOG_FEATURES, 'lgbm_mass+analog': MASS_FEATURES + ANALOG_FEATURES}
train_q = set(pos[pos > 0].index) & set(Q.loc[Q.regime.isin(['C1', 'C2']), 'query_id'])
TRAIN = F[F['query_id'].isin(train_q)]
qmeta = Q[['query_id', 'fold', 'true_connectivity_key']]
DISJ = connectivity_disjointness(qmeta[qmeta.query_id.isin(train_q)], qmeta)
display(DISJ)
assert (DISJ['n_overlap'] == 0).all(), 'connectivity leakage between ranker training and evaluation folds'
eval_ids = {f: set(g['query_id']) for f, g in Q.groupby('fold')}
MODELS = {}
for name, feats in VARIANTS.items():
    t0 = time.time()
    models, audit = fit_fold_models(TRAIN, feats, eval_query_ids_by_fold=eval_ids)
    F[f'score_{name}'] = predict_by_fold(models, F, feats).to_numpy()
    MODELS[name] = (models, feats)
    pq = per_query_ranks(F, f'score_{name}', Q)
    PER_QUERY[name] = pq
    for reg, g in pq.groupby('regime'):
        ROWS.append({'variant': name, 'kind': 'lambdarank', 'regime': reg, **ranking_metrics(g)})
    print(f'{name}: {time.time() - t0:.0f}s')
ABLATION = pd.DataFrame(ROWS)
display(ABLATION.pivot(index='variant', columns='regime', values='mrr_at_25').round(4))

In [ ]:
# PROVISIONAL composite (raw regime metrics stay the primary numbers)
COMPOSITE = []
for name, g in ABLATION.groupby('variant'):
    c = composite_mrr(dict(zip(g['regime'], g['mrr_at_25'])), CFG['validation']['composite_weights_PROVISIONAL'])
    COMPOSITE.append({'variant': name, 'composite_mrr_at_25_PROVISIONAL': c['composite_mrr_at_25'], **{f'{r}_mrr25': v for r, v in c['regime_mrr_at_25'].items()}})
COMPOSITE = pd.DataFrame(COMPOSITE)
display(COMPOSITE)
from casmi.validation.reporting import add_strata, stratified_table
best = 'lgbm_mass+analog'
STRAT = stratified_table(add_strata(PER_QUERY[best].rename(columns={'n_candidates': 'pool_size'}), Q, CFG['validation']['mass_bins_da'],
                                    CFG['validation']['pool_size_bins']), ranking_metrics, min_queries=20)
display(STRAT[STRAT.regime == 'C2'].head(40))

In [ ]:
from casmi.viz import plots_v2
OUT = P.reports_dir / 'analog_baseline' / RUN_TAG; OUT.mkdir(parents=True, exist_ok=True)
plots_v2.channel_ablation(ABLATION[ABLATION.regime.isin(['C1', 'C2'])], metric='mrr_at_25').savefig(OUT / 'ablation_mrr25.png', dpi=120)
plots_v2.channel_ablation(ABLATION[ABLATION.regime.isin(['C1', 'C2'])], metric='hit_at_1').savefig(OUT / 'ablation_hit1.png', dpi=120)
plots_v2.truth_rank_distribution(PER_QUERY[best]).savefig(OUT / 'truth_rank.png', dpi=120)

In [ ]:
from casmi.validation.reporting import save_report
from casmi.workspace.experiments import log_experiment
from casmi.workspace.artifacts import git_commit
# checkpoints (fold models) + predictions (top-25 per query) + reports
for name, (models, feats) in MODELS.items():
    save_fold_models(models, P.checkpoints_dir / 'analog_ranker' / RUN_TAG / name, feats, LGBM_PARAMS, sorted(train_q),
                     extra={'notebook': NOTEBOOK, 'feature_schema': feature_schema(feats), 'regime_signature': REG_META.get('signature')})
PRED = P.predictions_dir / 'analog_baseline' / RUN_TAG; PRED.mkdir(parents=True, exist_ok=True)
from casmi.ranking.rank_eval import rank_by_keys
r = rank_by_keys(F[['query_id', 'fold', 'regime', 'candidate_id', 'connectivity_key', 'is_true_candidate', 'abs_mass_error_ppm', f'score_{best}']],
                 [(f'score_{best}', False), ('abs_mass_error_ppm', True), ('candidate_id', True)])
r[r['rank'] <= 25].to_parquet(PRED / f'{best}_top25.parquet', index=False)
save_report(ABLATION, {'notebook': NOTEBOOK, 'run_tag': RUN_TAG, 'quick_run_cap': QUICK_RUN_MAX_QUERIES_PER_FOLD,
                       'composite_PROVISIONAL': COMPOSITE.to_dict(orient='records'), 'runs': RUNS.to_dict(orient='records'),
                       'regime_c2_mode': REG_META['c2_mode']}, OUT, 'analog_baseline_metrics')
STRAT.to_parquet(OUT / 'stratified_c1_c2.parquet', index=False)
write_metadata(OUT, NOTEBOOK, config={'analog': CFG['analog']}, seed=SEED, repo_root=REPO_ROOT, model_name='lightgbm_lambdarank',
               feature_list=MASS_FEATURES + ANALOG_FEATURES, schema_version=feature_schema()['schema_version'],
               input_paths={'regimes': P.processed_dir / 'validation_regimes.parquet', 'bundle': P.bundle_dir, 'universe': UNIVERSE_ROOT})
gc = git_commit(REPO_ROOT)
for _, c in COMPOSITE.iterrows():
    h1 = ABLATION[(ABLATION.variant == c['variant']) & (ABLATION.regime == 'C2')]['hit_at_1']
    log_experiment(P.experiments_dir, f"{RUN_TAG}:{c['variant']}", 'analog baseline (notebook 14)', ['mass', 'analog'], config_path=CFG['_config_path'],
                   git_commit=gc, C1_MRR25=c.get('C1_mrr25'), C2_MRR25=c.get('C2_mrr25'), C3_MRR25=c.get('C3_mrr25'),
                   composite_MRR25=c['composite_mrr_at_25_PROVISIONAL'], Hit1=float(h1.iloc[0]) if len(h1) else None,
                   notes='quick run' if QUICK_RUN_MAX_QUERIES_PER_FOLD else 'full')
print('saved under', OUT, PRED, P.checkpoints_dir / 'analog_ranker' / RUN_TAG)